# 02 — Cross-Source Validation

This notebook checks whether the three FYTA data sources tell a consistent story.

1. **Image-label validation** — compare each image label with the previous **3 hours** of sensor data using the median and Mann–Kendall trend, while attaching recent context logs.
2. **Feature triangulation** — start from a logged care event, compare sensor behaviour in the **60 minutes before and after the event**, and look for a related image within the next **24 hours**.
3. **Optional text characterization** — derive a few structured features from the free-text notes.

The analysis is intentionally rule-based and interpretable; the signals are treated as consistency checks, not causal proof.


In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from scipy.stats import norm

DATA_DIR = Path("src/data")

plant_df = pd.read_csv(DATA_DIR / "plant_feature_table.csv")
image_df = pd.read_csv(DATA_DIR / "fyta_images.csv")
context_df = pd.read_csv(DATA_DIR / "fyta_contextual.csv")

plant_df["timestamp"] = pd.to_datetime(
    plant_df["timestamp"], format="mixed", errors="coerce"
)

image_df["captured_at"] = pd.to_datetime(
    image_df["captured_at"], format="mixed", errors="coerce"
)

context_df["created_at"] = pd.to_datetime(
    context_df["created_at"].astype(str).str.strip(),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Plant rows:", len(plant_df))
print("Image rows:", len(image_df))
print("Context rows:", len(context_df))


Plant rows: 11903
Image rows: 28
Context rows: 49


The datasets are aligned by `user_plant_id` and timestamp.

Context timestamps are parsed explicitly as `YYYY-MM-DD HH:MM:SS` to preserve the June dates correctly before validation and triangulation.


# 1. Image-Label Validation / Weak Supervision

**Goal:** Check whether each image-model label is consistent with recent sensor behaviour.

### Validation flow

**Image label**
→ **Previous 3 h sensor window**
→ **Median + Mann–Kendall trend**
→ **Compare with plant-specific Q25/Q75**
→ **Attach recent 24 h context logs**
→ **Agreement / Disagreement / Uncertain**

Only labels that can be meaningfully validated with the available sensors are evaluated:

- `Water deficiency`
- `Water excess and/or uneven watering`
- `Nutrient deficiency`

The rules are used as consistency checks, not as proof that the image label is biologically correct.

In [2]:
TARGET_ISSUES = {
    "Water deficiency",
    "Water excess and/or uneven watering",
    "Nutrient deficiency"
}


def extract_image_label(value):
    prediction = json.loads(value) if isinstance(value, str) else value
    health = prediction.get("health_assessment", {})
    diseases = health.get("diseases", [])

    candidates = [
        x for x in diseases
        if x.get("name") in TARGET_ISSUES
    ]

    if not candidates:
        return None

    top = max(candidates, key=lambda x: x.get("probability", 0))
    return top.get("name")


image_df["image_label"] = image_df["prediction"].apply(extract_image_label)

validation_df = image_df[
    image_df["image_label"].notna()
].copy()

display(
    validation_df[
        ["image_id", "user_plant_id", "captured_at", "image_label"]
    ].head()
)


,image_id,user_plant_id,captured_at,image_label
0,IMG-1001-202606041630,UP-1001,2026-06-04 16:30:00,Water deficiency
1,IMG-1001-202606111345,UP-1001,2026-06-11 13:45:00,Water excess and/or uneven watering
2,IMG-1001-202606161515,UP-1001,2026-06-16 15:15:00,Water excess and/or uneven watering
3,IMG-1002-202606010915,UP-1002,2026-06-01 09:15:00,Water excess and/or uneven watering
4,IMG-1002-202606041715,UP-1002,2026-06-04 17:15:00,Water deficiency


The image JSON is reduced to the three issue labels relevant to the available moisture and EC measurements.

In [3]:
SENSOR_WINDOW = pd.Timedelta(hours=3)
MIN_MK_POINTS = 8


def mann_kendall(values, alpha=0.05):
    """Return monotonic trend and p-value using the Mann–Kendall test."""
    x = pd.Series(values).dropna().to_numpy()
    n = len(x)

    if n < MIN_MK_POINTS:
        return "insufficient", np.nan

    s = sum(
        np.sign(x[j] - x[i])
        for i in range(n - 1)
        for j in range(i + 1, n)
    )

    _, counts = np.unique(x, return_counts=True)
    var_s = (
        n * (n - 1) * (2 * n + 5)
        - np.sum(counts * (counts - 1) * (2 * counts + 5))
    ) / 18

    if var_s == 0:
        z = 0.0
    elif s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p_value = 2 * (1 - norm.cdf(abs(z)))

    if p_value >= alpha:
        return "stable", p_value

    return ("increasing" if z > 0 else "decreasing"), p_value


def recent_sensor_summary(row):
    history = plant_df[
        (plant_df["user_plant_id"] == row["user_plant_id"])
        & (plant_df["timestamp"] <= row["captured_at"])
        & (plant_df["timestamp"] >= row["captured_at"] - SENSOR_WINDOW)
    ].sort_values("timestamp")

    moisture_trend, moisture_p = mann_kendall(
        history["soil_moisture_vwc_clean"]
    )
    ec_trend, ec_p = mann_kendall(
        history["ec_us_cm_clean"]
    )

    return pd.Series({
        "moisture_median_3h": history["soil_moisture_vwc_clean"].median(),
        "ec_median_3h": history["ec_us_cm_clean"].median(),
        "moisture_mk_trend": moisture_trend,
        "moisture_mk_p": moisture_p,
        "ec_mk_trend": ec_trend,
        "ec_mk_p": ec_p
    })


sensor_summary = validation_df.apply(recent_sensor_summary, axis=1)
validation_df = pd.concat(
    [validation_df.reset_index(drop=True), sensor_summary.reset_index(drop=True)],
    axis=1
)


For each image, sensor readings from the previous **3 hours** are summarized in two ways: the median describes the recent level, while Mann–Kendall describes the temporal direction. Using both prevents the decision from depending on one isolated reading.


In [4]:
def recent_log_types(row):
    logs = context_df[
        (context_df["user_plant_id"] == row["user_plant_id"])
        & (context_df["created_at"] <= row["captured_at"])
        & (context_df["created_at"] >= row["captured_at"] - pd.Timedelta(hours=24))
    ]

    if logs.empty:
        return None

    return ", ".join(
        sorted(logs["log_type"].dropna().astype(str).unique())
    )


validation_df["recent_log_types"] = validation_df.apply(
    recent_log_types, axis=1
)


Recent contextual evidence is kept simple: only the existing `log_type` values from the previous 24 hours are attached to each image.

In [5]:
plant_thresholds = (
    plant_df
    .groupby("user_plant_id")
    .agg(
        moisture_low=("soil_moisture_vwc_clean", lambda x: x.quantile(0.25)),
        moisture_high=("soil_moisture_vwc_clean", lambda x: x.quantile(0.75)),
        ec_low=("ec_us_cm_clean", lambda x: x.quantile(0.25)),
        ec_high=("ec_us_cm_clean", lambda x: x.quantile(0.75))
    )
    .reset_index()
)

validation_df = validation_df.merge(
    plant_thresholds,
    on="user_plant_id",
    how="left"
)


Each plant is compared with its own sensor history. The 25th and 75th percentiles are used only as relative screening references, not biological thresholds.

In [6]:
def cross_validate(row):
    label = row["image_label"]
    moisture = row["moisture_median_3h"]
    trend = row["moisture_mk_trend"]
    q25 = row["moisture_low"]
    q75 = row["moisture_high"]

    # Water deficiency
    if label == "Water deficiency":
        if moisture <= q25:
            return "agreement"
        elif moisture >= q75:
            return "disagreement"
        elif trend == "decreasing":
            return "agreement"
        elif trend == "increasing":
            return "disagreement"
        else:
            return "uncertain"

    # Water excess / uneven watering
    if label == "Water excess and/or uneven watering":
        if moisture >= q75:
            return "agreement"
        elif moisture <= q25:
            return "disagreement"
        elif trend == "increasing":
            return "agreement"
        elif trend == "decreasing":
            return "disagreement"
        else:
            return "uncertain"

    return "uncertain"


# Apply validation
validation_df["cross_validation"] = validation_df.apply(
    cross_validate,
    axis=1
)

# Add species
plant_species = (
    plant_df[["user_plant_id", "species"]]
    .drop_duplicates()
)

validation_df = validation_df.merge(
    plant_species,
    on="user_plant_id",
    how="left"
)

# Summary
print(validation_df["cross_validation"].value_counts())

# Clean validation table
display(
    validation_df[
        [
            "user_plant_id",
            "species",
            "captured_at",
            "image_label",
            "moisture_median_3h",
            "moisture_low",        # Q25
            "moisture_high",       # Q75
            "moisture_mk_trend",
            "recent_log_types",
            "cross_validation"
        ]
    ].rename(
        columns={
            "moisture_low": "moisture_q25",
            "moisture_high": "moisture_q75"
        }
    )
)

cross_validation
agreement       22
disagreement     4
uncertain        2
Name: count, dtype: int64


,user_plant_id,species,captured_at,image_label,moisture_median_3h,moisture_q25,moisture_q75,moisture_mk_trend,recent_log_types,cross_validation
0,UP-1001,Monstera deliciosa,2026-06-04 16:30:00,Water deficiency,22.895,27.7050,33.95625,stable,watering,agreement
1,UP-1001,Monstera deliciosa,2026-06-11 13:45:00,Water excess and/or uneven watering,35.085,27.7050,33.95625,decreasing,NaN,agreement
2,UP-1001,Monstera deliciosa,2026-06-16 15:15:00,Water excess and/or uneven watering,34.930,27.7050,33.95625,decreasing,watering,agreement
3,UP-1002,Ficus lyrata,2026-06-01 09:15:00,Water excess and/or uneven watering,37.840,28.7975,37.30000,stable,NaN,agreement
4,UP-1002,Ficus lyrata,2026-06-04 17:15:00,Water deficiency,25.250,28.7975,37.30000,decreasing,NaN,agreement
5,UP-1002,Ficus lyrata,2026-06-07 09:00:00,Water deficiency,24.910,28.7975,37.30000,decreasing,NaN,agreement
6,UP-1002,Ficus lyrata,2026-06-10 12:15:00,Water deficiency,31.760,28.7975,37.30000,decreasing,watering,agreement
7,UP-1002,Ficus lyrata,2026-06-11 18:30:00,Water excess and/or uneven watering,33.880,28.7975,37.30000,stable,NaN,uncertain
8,UP-1002,Ficus lyrata,2026-06-15 13:30:00,Water excess and/or uneven watering,34.690,28.7975,37.30000,decreasing,NaN,disagreement
9,UP-1003,Zamioculcas zamiifolia,2026-06-02 11:45:00,Water excess and/or uneven watering,26.960,25.5200,41.98750,decreasing,watering,disagreement


The decision combines **level and direction**. A clearly low/high plant-relative median is strong evidence; when the median lies between the quartiles, the Mann–Kendall trend is used as supporting evidence. If neither provides a clear signal, the image remains `uncertain`. Recent context logs are retained for interpretation rather than forced into the label decision.


# 2. Feature Triangulation

**Goal:** Check whether a logged care event is supported by sensor behaviour and/or a nearby image.

### Triangulation flow

**Context event**
→ **±60 min sensor window**
→ **Compare before vs after median**
→ **Check expected sensor response**
→ **Check nearest related image within 24 h**
→ **Triangulated / Not triangulated / Unavailable**

### Sensor rules

- `watering` → soil moisture increases
- `fertilising` → EC **or** soil moisture increases
- `light` → PAR moves in the direction implied by the note
- `repotting` → no deterministic sensor rule is used

An event is marked **triangulated** when either the sensor response or a related image supports the logged event.

In [7]:
EVENT_WINDOW = pd.Timedelta(minutes=60)
IMAGE_WINDOW = pd.Timedelta(hours=24)


def nearest_post_event_image(plant_id, event_time):
    images = image_df[
        (image_df["user_plant_id"] == plant_id)
        & (image_df["captured_at"] >= event_time)
        & (image_df["captured_at"] <= event_time + IMAGE_WINDOW)
    ].copy()

    if images.empty:
        return pd.Series({
            "nearby_image_label": None,
            "image_time": pd.NaT,
            "image_delay_hours": np.nan
        })

    images["distance"] = images["captured_at"] - event_time
    nearest = images.sort_values("distance").iloc[0]

    return pd.Series({
        "nearby_image_label": nearest["image_label"],
        "image_time": nearest["captured_at"],
        "image_delay_hours": nearest["distance"].total_seconds() / 3600
    })


def light_expected_direction(text):
    text = str(text).lower()

    if "moved back" in text or "too much direct sun" in text:
        return "decreasing"

    if "closer to window" in text or "toward the light" in text:
        return "increasing"

    return None


def median_change_support(before_median, after_median, expected_direction):
    if pd.isna(before_median) or pd.isna(after_median):
        return False

    if expected_direction == "increasing":
        return after_median > before_median

    if expected_direction == "decreasing":
        return after_median < before_median

    return False


def event_sensor_summary(event):
    event_time = event["created_at"]

    plant_history = plant_df[
        plant_df["user_plant_id"] == event["user_plant_id"]
    ].sort_values("timestamp")

    before = plant_history[
        (plant_history["timestamp"] >= event_time - EVENT_WINDOW)
        & (plant_history["timestamp"] < event_time)
    ]

    after = plant_history[
        (plant_history["timestamp"] >= event_time)
        & (plant_history["timestamp"] <= event_time + EVENT_WINDOW)
    ]

    # Watering: moisture should increase
    if event["log_type"] == "watering":
        moisture_before = before["soil_moisture_vwc_clean"].median()
        moisture_after = after["soil_moisture_vwc_clean"].median()

        moisture_support = median_change_support(
            moisture_before,
            moisture_after,
            "increasing"
        )

        return pd.Series({
            "sensor_variable": "soil_moisture_vwc_clean",
            "expected_direction": "increasing",
            "before_median": moisture_before,
            "after_median": moisture_after,
            "secondary_before_median": np.nan,
            "secondary_after_median": np.nan,
            "sensor_support": moisture_support
        })

    # Fertilising: EC OR moisture increase is sufficient
    if event["log_type"] == "fertilising":
        ec_before = before["ec_us_cm_clean"].median()
        ec_after = after["ec_us_cm_clean"].median()

        moisture_before = before["soil_moisture_vwc_clean"].median()
        moisture_after = after["soil_moisture_vwc_clean"].median()

        ec_support = median_change_support(
            ec_before,
            ec_after,
            "increasing"
        )

        moisture_support = median_change_support(
            moisture_before,
            moisture_after,
            "increasing"
        )

        return pd.Series({
            "sensor_variable": "ec_us_cm_clean OR soil_moisture_vwc_clean",
            "expected_direction": "increasing",
            "before_median": ec_before,
            "after_median": ec_after,
            "secondary_before_median": moisture_before,
            "secondary_after_median": moisture_after,
            "sensor_support": ec_support or moisture_support
        })

    # Light: PAR direction depends on the user note
    if event["log_type"] == "light":
        expected = light_expected_direction(event["text"])

        par_before = before["light_par_clean"].median()
        par_after = after["light_par_clean"].median()

        par_support = median_change_support(
            par_before,
            par_after,
            expected
        )

        return pd.Series({
            "sensor_variable": "light_par_clean",
            "expected_direction": expected,
            "before_median": par_before,
            "after_median": par_after,
            "secondary_before_median": np.nan,
            "secondary_after_median": np.nan,
            "sensor_support": par_support
        })

    # Repotting: no single deterministic sensor response
    return pd.Series({
        "sensor_variable": None,
        "expected_direction": None,
        "before_median": np.nan,
        "after_median": np.nan,
        "secondary_before_median": np.nan,
        "secondary_after_median": np.nan,
        "sensor_support": False
    })


events = context_df.copy()

sensor_summary = events.apply(
    event_sensor_summary,
    axis=1
)

triangulation_df = pd.concat(
    [
        events.reset_index(drop=True),
        sensor_summary.reset_index(drop=True)
    ],
    axis=1
)

image_summary = triangulation_df.apply(
    lambda row: nearest_post_event_image(
        row["user_plant_id"],
        row["created_at"]
    ),
    axis=1
)

triangulation_df = pd.concat(
    [
        triangulation_df.reset_index(drop=True),
        image_summary.reset_index(drop=True)
    ],
    axis=1
)


The sensor check uses only the **median before vs median after the event** within a single ±60-minute window.

For an event at `10:00`:

- before median: `09:00–09:59`
- after median: `10:00–11:00`

This reduces sensitivity to individual noisy readings while keeping the response close to the logged event. For fertilising, either an EC increase or a moisture increase is accepted as sensor support.


In [8]:
def related_image_signal(row):
    label = row["nearby_image_label"]

    if pd.isna(label):
        return False

    if row["log_type"] == "watering":
        return label in {
            "Water deficiency",
            "Water excess and/or uneven watering"
        }

    if row["log_type"] == "fertilising":
        return label == "Nutrient deficiency"

    # No direct image label is available for light or repotting.
    return False


triangulation_df["related_image_signal"] = triangulation_df.apply(
    related_image_signal,
    axis=1
)


def triangulation_status(row):
    if row["log_type"] == "repotting":
        return "unavailable"

    # Either source can support the event.
    if row["sensor_support"] or row["related_image_signal"]:
        return "triangulated"

    # If both before/after medians are unavailable, there is not enough sensor evidence.
    if pd.isna(row["before_median"]) or pd.isna(row["after_median"]):
        return "unavailable"

    return "not_triangulated"


triangulation_df["triangulation_status"] = triangulation_df.apply(
    triangulation_status,
    axis=1
)

print("Triangulation summary:")
print(triangulation_df["triangulation_status"].value_counts())

display(
    triangulation_df[
        [
            "log_id",
            "user_plant_id",
            "created_at",
            "log_type",
            "sensor_variable",
            "expected_direction",
            "before_median",
            "after_median",
            "secondary_before_median",
            "secondary_after_median",
            "sensor_support",
            "nearby_image_label",
            "related_image_signal",
            "triangulation_status"
        ]
    ]
)


Triangulation summary:
triangulation_status
not_triangulated    23
triangulated        22
unavailable          4
Name: count, dtype: int64


,log_id,user_plant_id,created_at,log_type,sensor_variable,expected_direction,before_median,after_median,secondary_before_median,secondary_after_median,sensor_support,nearby_image_label,related_image_signal,triangulation_status
0,LOG-00000,UP-1001,2026-06-01 09:00:00,watering,soil_moisture_vwc_clean,increasing,36.5000,36.125,NaN,NaN,False,NaT,False,not_triangulated
1,LOG-00001,UP-1001,2026-06-03 14:00:00,fertilising,ec_us_cm_clean OR soil_moisture_vwc_clean,increasing,692.7500,700.500,24.625,24.475,True,NaT,False,triangulated
2,LOG-00002,UP-1001,2026-06-04 07:30:00,watering,soil_moisture_vwc_clean,increasing,23.0450,23.100,NaN,NaN,True,Water deficiency,True,triangulated
3,LOG-00003,UP-1001,2026-06-07 11:30:00,watering,soil_moisture_vwc_clean,increasing,29.9150,29.555,NaN,NaN,False,NaT,False,not_triangulated
4,LOG-00004,UP-1001,2026-06-12 17:30:00,watering,soil_moisture_vwc_clean,increasing,28.9425,28.690,NaN,NaN,False,NaT,False,not_triangulated
5,LOG-00005,UP-1001,2026-06-15 19:45:00,watering,soil_moisture_vwc_clean,increasing,29.8425,29.365,NaN,NaN,False,Water excess and/or uneven watering,True,triangulated
6,LOG-00006,UP-1001,2026-06-20 08:30:00,watering,soil_moisture_vwc_clean,increasing,33.0700,32.865,NaN,NaN,False,NaT,False,not_triangulated
7,LOG-00007,UP-1001,2026-06-21 11:00:00,light,light_par_clean,increasing,282.4500,323.350,NaN,NaN,True,NaT,False,triangulated
8,LOG-00008,UP-1002,2026-06-01 13:30:00,watering,soil_moisture_vwc_clean,increasing,35.2550,34.060,NaN,NaN,False,NaT,False,not_triangulated
9,LOG-00009,UP-1002,2026-06-01 20:00:00,fertilising,ec_us_cm_clean OR soil_moisture_vwc_clean,increasing,682.5000,701.000,33.040,32.280,True,NaT,False,triangulated


The triangulation decision is intentionally simple:

- **`triangulated`** — the ±60-minute sensor medians show the expected response **or** a related image is found within the following 24 hours.
- **`not_triangulated`** — neither source provides supporting evidence.
- **`unavailable`** — the required sensor evidence is missing or no clear event-specific response is defined.  
  For repotting, this could be extended in future using a **substrate disturbance signal**, for example by detecting unusually large changes in soil moisture or EC over a wider time window around the event.

The image is supporting evidence rather than a mandatory requirement. This means an event can still be triangulated from the sensor response when no nearby image exists.


# 3. Optional Text Characterization

The previous tasks use `log_type` directly because it is already structured and sufficient for the main analysis.

As an optional stretch, the free-text field can still be converted into additional structured attributes such as dry soil, drooping, watering amount, direct sun, or fertiliser type.

In [9]:
def extract_text_features(text):
    text = str(text).lower()

    amount = re.search(r"(\d+)\s*ml", text)

    return {
        "dry_soil": "dry" in text,
        "drooping": "droop" in text,
        "good_soak": "good soak" in text,
        "bottom_watered": "bottom-watered" in text,
        "direct_sun": "direct sun" in text,
        "npk_feed": "npk" in text,
        "seaweed_feed": "seaweed" in text,
        "watering_amount_ml": float(amount.group(1)) if amount else np.nan
    }


text_features = (
    context_df["text"]
    .apply(extract_text_features)
    .apply(pd.Series)
)

context_text_df = pd.concat(
    [context_df, text_features], axis=1
)

display(context_text_df.head(10))


,log_id,user_plant_id,log_type,text,created_at,dry_soil,drooping,good_soak,bottom_watered,direct_sun,npk_feed,seaweed_feed,watering_amount_ml
0,LOG-00000,UP-1001,watering,Watered ~200ml,2026-06-01 09:00:00,False,False,False,False,False,False,False,200.0
1,LOG-00001,UP-1001,fertilising,Diluted seaweed feed,2026-06-03 14:00:00,False,False,False,False,False,False,True,NaN
2,LOG-00002,UP-1001,watering,"Watered a bit, leaves drooping",2026-06-04 07:30:00,False,True,False,False,False,False,False,NaN
3,LOG-00003,UP-1001,watering,"Watered a bit, leaves drooping",2026-06-07 11:30:00,False,True,False,False,False,False,False,NaN
4,LOG-00004,UP-1001,watering,"Top soil was dry, watered",2026-06-12 17:30:00,True,False,False,False,False,False,False,NaN
5,LOG-00005,UP-1001,watering,Gave it a good soak,2026-06-15 19:45:00,False,False,True,False,False,False,False,NaN
6,LOG-00006,UP-1001,watering,Bottom-watered today,2026-06-20 08:30:00,False,False,False,True,False,False,False,NaN
7,LOG-00007,UP-1001,light,Moved closer to window,2026-06-21 11:00:00,False,False,False,False,False,False,False,NaN
8,LOG-00008,UP-1002,watering,Quick water,2026-06-01 13:30:00,False,False,False,False,False,False,False,NaN
9,LOG-00009,UP-1002,fertilising,Added liquid fertiliser (half dose),2026-06-01 20:00:00,False,False,False,False,False,False,False,NaN


Because the contextual vocabulary is small and repetitive, a simple rule-based parser is sufficient here. An LLM API would only become useful with larger, more varied, multilingual, or ambiguous free-text logs.

## Conclusion

The notebook keeps the two tasks separate and interpretable:

**Image-label validation:** previous 3 h sensor median + Mann–Kendall trend + recent context.

**Feature triangulation:** ±60 min before/after sensor medians + nearest related image within 24 h.

For triangulation, watering is checked with soil moisture, fertilising with **EC OR moisture**, and light with PAR. An event is considered triangulated when either the sensor response or related image evidence supports the contextual log.
